# Inferencia retrospectiva e produto operacional

Este notebook demonstra a cadeia operacional a partir de uma sequencia real de capturas PNG do radar Sumaré: selecao de evento forte pelo AlertaRio, agregacao em frames de 15 minutos, remocao do menu lateral e preparacao do tensor de entrada.

A previsao numerica exige um checkpoint treinado com o mesmo recorte canônico declarado em `configs/radar_sumare_capture_v1.json`. Checkpoints legados sao rejeitados deliberadamente.

## Contrato temporal

Cada frame do modelo representa uma janela de 15 minutos, agregada por maximo pixel a pixel a partir de PNGs de aproximadamente 2 minutos. A entrada tem cinco frames passados (75 minutos); a saida tem cinco horizontes futuros, de T+15 a T+75 minutos.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from nowcasting.operational_inference import prepare_radar_input
from nowcasting.radar_capture import load_capture_config

ROOT = Path.cwd()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parent.parent

CAPTURE_ROOT = ROOT / 'data' / 'radar_sumare'
EVENTS_PATH = ROOT / 'outputs' / 'analysis' / 'radar_sumare' / 'operational_events_v1' / 'events.json'
CAPTURE_CONFIG_PATH = ROOT / 'configs' / 'radar_sumare_capture_v1.json'

events = json.loads(EVENTS_PATH.read_text(encoding='utf-8'))
pd.DataFrame([
    {
        'evento_utc': event['event_timestamp_utc'],
        'estacao': event['station_name'],
        'm15_mm': event['m15_mm_15min'],
    }
    for event in events
]).sort_values('m15_mm', ascending=False)

In [ ]:
EVENT_INDEX = 0
event = events[EVENT_INDEX]

print('Evento:', event['event_timestamp_utc'])
print('Estacao com maior acumulado:', event['station_name'])
print('m15:', event['m15_mm_15min'], 'mm/15 min')

coverage = pd.DataFrame([
    {
        'grupo': group,
        'timestamp_utc': bucket['timestamp_utc'],
        'pngs_no_bucket': len(bucket['png_files']),
    }
    for group in ('input', 'target')
    for bucket in event[group]
])
coverage

In [ ]:
capture_config = load_capture_config(CAPTURE_CONFIG_PATH)
# Esta configuracao descreve somente o formato do tensor de um modelo futuro.
# O checkpoint real fornece os mesmos campos durante a inferencia.
model_configuration = {'step': 5, 'crop': {'enabled': False}}

X = prepare_radar_input(
    event,
    CAPTURE_ROOT,
    capture_config,
    model_configuration,
    width=128,
    height=128,
)
print('X:', X.shape, X.dtype)  # [batch, canais RGB, tempo, altura, largura]

In [ ]:
fig, axes = plt.subplots(1, X.shape[2], figsize=(15, 3))
for index, axis in enumerate(axes):
    axis.imshow(X[0, :, index].transpose(1, 2, 0))
    axis.set_title(event['input'][index]['timestamp_utc'][11:16] + ' UTC')
    axis.axis('off')
fig.suptitle('Cinco frames de entrada apos recorte canônico e agregacao de 15 min')
plt.show()

## Mock operacional para demonstracao

Antes de haver um checkpoint treinado com este recorte canônico, o produto abaixo permite discutir o fluxo operacional com a equipe meteorológica. Ele mostra os cinco frames recentes de radar na ROI cropada das estações e uma previsão determinística de **persistência nas estações** para T+15 a T+75 min. Também grava um mapa Folium com uma camada selecionável por horizonte. O mapa não usa tiles externos por padrão, para permanecer reproduzível em redes sem acesso ou sem credenciais cartográficas; a base municipal local do IBGE é incorporada como GeoJSON.

O mock não executa um modelo treinado e não produz campo contínuo de chuva: a supervisão atual é esparsa, somente nos pixels das estações. As observações futuras incluídas nos CSVs servem exclusivamente para avaliação retrospectiva.

In [ ]:
from IPython.display import Image as DisplayImage, display

from nowcasting.operational_mock import run_mock

ALERTARIO_ROOT = ROOT / 'data' / 'pluviometricos_alertario'
MAPPING_PATH = ROOT / 'configs' / 'mapeamento_pixel_estacao_alertario_capture_crop_v1.csv'
GEOGRAPHIC_GRID_PATH = ROOT / 'data' / 'sumare_radar_latlon_grid.npz'
MUNICIPAL_BASEMAP_PATH = ROOT / 'outputs' / 'analysis' / 'geospatial' / 'ibge' / 'rj_municipios_roi_cropada_2024.geojson'
mock_output_dir = ROOT / 'outputs' / 'operational_demo_mock' / event['event_timestamp_utc'].replace(':', '-')

mock_summary = run_mock(
    event=event,
    capture_root=CAPTURE_ROOT,
    capture_config=capture_config,
    alertario_root=ALERTARIO_ROOT,
    mapping=pd.read_csv(MAPPING_PATH),
    output_dir=mock_output_dir,
    width=128,
    height=128,
    max_m15=175.0,
    crop_stations=True,
    crop_margin_pixels=20,
    geographic_grid_path=GEOGRAPHIC_GRID_PATH,
    show_station_names=False,
    map_tiles='none',
    municipal_basemap_path=MUNICIPAL_BASEMAP_PATH,
)
mock_summary

display(DisplayImage(filename=mock_output_dir / 'mock_product.png'))

## Executar uma previsao

Depois de treinar um modelo com os memmaps gerados por `nowcasting-build-radar --capture-config ...`, defina o caminho do checkpoint abaixo. O CLI verifica que a geometria gravada no checkpoint e identica a configuracao de captura antes de produzir `forecast.npz`.

In [ ]:
CHECKPOINT = None  # Path('/caminho/para/iteration_1_best.pt')

if CHECKPOINT is None:
    print('Aguardando checkpoint treinado com o recorte canonico.')
else:
    output_dir = ROOT / 'outputs' / 'operational_demo' / event['event_timestamp_utc'].replace(':', '-')
    command = (
        f'nowcasting-infer-operational --checkpoint {CHECKPOINT} --events {EVENTS_PATH} '
        f'--event-index {EVENT_INDEX} --capture-root {CAPTURE_ROOT} '
        f'--capture-config {CAPTURE_CONFIG_PATH} --output-dir {output_dir}'
    )
    print(command)